In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', 100)
sns.set_style('darkgrid')

synthetic_path = '../data/processed/synthetic_transactions.csv'
ieee_path = '../data/processed/ieee_cis_sample.csv'

if not os.path.exists(synthetic_path):
    raise FileNotFoundError(f"Could not find {synthetic_path} — check you're running Jupyter from the notebooks/ folder.")
if not os.path.exists(ieee_path):
    raise FileNotFoundError(f"Could not find {ieee_path} — check you're running Jupyter from the notebooks/ folder.")

df_synthetic = pd.read_csv(synthetic_path)
df_ieee = pd.read_csv(ieee_path)

print("Loaded successfully.")


In [ ]:
print("=== SYNTHETIC ===")
print("Shape:", df_synthetic.shape)
print("Fraud ratio (label.mean()):", df_synthetic['label'].mean())

print("\n=== IEEE-CIS ===")
print("Shape:", df_ieee.shape)
print("Fraud ratio (isFraud.mean()):", df_ieee['isFraud'].mean())

In [ ]:
fraud_ratios = pd.Series({
    'Synthetic': df_synthetic['label'].mean(),
    'IEEE-CIS': df_ieee['isFraud'].mean()
})

plt.figure(figsize=(6, 4))
fraud_ratios.plot(kind='bar', color=['steelblue', 'darkorange'])
plt.ylabel('Fraud Ratio')
plt.title('Fraud Ratio: Synthetic vs IEEE-CIS')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Transaction amount distribution — Synthetic

What to look for: if the synthetic data generator is doing its job, fraud transactions
should show a visibly different amount distribution than legit ones (e.g. shifted toward
higher or lower amounts, different spread).
If the two curves look nearly identical, that's a signal the synthetic fraud labels aren't carrying real distributional signal 
worth flagging for Phase 2 feature engineering.

In [ ]:
plt.figure(figsize=(8, 5))
legit = df_synthetic[df_synthetic['label'] == 0]['amount']
fraud = df_synthetic[df_synthetic['label'] == 1]['amount']

plt.hist(legit, bins=50, alpha=0.5, label='Legit', color='steelblue')
plt.hist(fraud, bins=50, alpha=0.5, label='Fraud', color='crimson')
plt.xscale('log')
plt.xlabel('Amount (log scale)')
plt.ylabel('Count')
plt.title('Synthetic: Transaction Amount Distribution (Fraud vs Legit)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
legit_ieee = df_ieee[df_ieee['isFraud'] == 0]['TransactionAmt']
fraud_ieee = df_ieee[df_ieee['isFraud'] == 1]['TransactionAmt']

plt.hist(legit_ieee, bins=50, alpha=0.5, label='Legit', color='steelblue')
plt.hist(fraud_ieee, bins=50, alpha=0.5, label='Fraud', color='crimson')
plt.xscale('log')
plt.xlabel('TransactionAmt (log scale)')
plt.ylabel('Count')
plt.title('IEEE-CIS: Transaction Amount Distribution (Fraud vs Legit)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
user_counts = df_synthetic['user_id'].value_counts()

plt.figure(figsize=(8, 5))
plt.hist(user_counts, bins=50, color='steelblue')
plt.xlabel('Transactions per user_id')
plt.ylabel('Number of users')
plt.title('Synthetic: Transaction Count Distribution per User')
plt.tight_layout()
plt.show()

print("Top 10 most active user_ids:")
print(user_counts.head(10))

In [ ]:
card_counts = df_ieee['card1'].value_counts()

plt.figure(figsize=(8, 5))
plt.hist(card_counts, bins=50, color='darkorange')
plt.xlabel('Transactions per card1')
plt.ylabel('Number of cards')
plt.title('IEEE-CIS: Transaction Count Distribution per card1')
plt.tight_layout()
plt.show()

print("Top 10 most active card1 values:")
print(card_counts.head(10))

In [ ]:
null_synthetic = (df_synthetic.isnull().mean() * 100).round(2).sort_values(ascending=False)
null_ieee = (df_ieee.isnull().mean() * 100).round(2).sort_values(ascending=False)

print("=== SYNTHETIC: Null % per column ===")
print(null_synthetic)

print("\n=== IEEE-CIS: Null % per column ===")
print(null_ieee)

## Summary (fill in after reviewing plots above)

- Fraud ratio comparison: ___
- Amount distribution differences (synthetic vs IEEE): ___
- Does synthetic fraud look distinguishable from legit, or too similar?: ___
- Null rate differences and what they imply for Phase 2 feature engineering: ___
- Transaction concentration (user_id / card1) — any users/cards standing out as high-velocity?: ___
- Anything surprising or concerning that changes the Phase 2 plan: ___